ESM-2 + DeepLoc: Investigating Protein Subcellular Localization

Introduction

This notebook is a controlled reconstruction of an earlier ESM-2 + DeepLoc protein subcellular localization project in which the validation performance unexpectedly reached F1 = 0.00 and accuracy = 0.00.

Rather than immediately changing the model or tuning hyperparameters, the goal of this notebook is to rebuild the pipeline step by step and identify what actually went wrong.

The overall pipeline is:

Protein sequence → ESM-2 → sequence representation → classification head → localization predictions

DeepLoc is treated as a multi-label classification problem, because a protein can belong to more than one cellular localization category. The model therefore predicts 10 independent localization labels:

* Cytoplasm
* Nucleus
* Extracellular
* Cell membrane
* Mitochondrion
* Plastid
* Endoplasmic reticulum
* Lysosome/Vacuole
* Golgi apparatus
* Peroxisome

The notebook first establishes that the data, labels, tokenization, ESM-2 representations, classifier, loss function and evaluation pipeline are working correctly. A frozen ESM-2 baseline is then compared with a fine-tuned ESM-2 model.

The purpose is therefore not simply to obtain a high score, but to understand why the previous pipeline failed and which corrections make the system work.

In [1]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer,AutoModel
from datasets import load_dataset

In [2]:
import torch
device= torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:",device)

Device: cuda


In [3]:
!pip install -q transformers datasets

In [5]:
model_name= "facebook/esm2_t6_8M_UR50D"

tokenizer= AutoTokenizer.from_pretrained(model_name)
model=AutoModel.from_pretrained(model_name)
model= model.to(device)

config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 31.4MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [6]:
print(model.config.hidden_size)
print(model.config.num_hidden_layers)

320
6


In [7]:
sequence="MKTLLV"

tokens= tokenizer(sequence,return_tensors="pt")
print(tokens["input_ids"])
print(tokens["input_ids"].shape)

tensor([[ 0, 20, 15, 11,  4,  4,  7,  2]])
torch.Size([1, 8])


In [10]:
outputs=model(
    input_ids=tokens["input_ids"].to(device),
    attention_mask=tokens["attention_mask"].to(device)
)
print(outputs.last_hidden_state.shape)

torch.Size([1, 8, 320])


In [11]:
M_embedding= outputs.last_hidden_state[0,1]
print(M_embedding.shape)
print(M_embedding[:10])

torch.Size([320])
tensor([ 0.2585,  0.1962,  0.1706,  0.5539, -0.1469, -0.1948, -0.4299, -0.3192,
        -0.1021,  0.0759], device='cuda:0', grad_fn=<SliceBackward0>)


In [12]:
cls_embedding=outputs.last_hidden_state[0,0]
print(cls_embedding.shape)
print(cls_embedding[:10])

torch.Size([320])
tensor([ 0.1438,  0.4320,  0.3049,  0.3545, -0.2643, -0.2227, -0.6780, -0.0072,
        -0.2449, -0.7084], device='cuda:0', grad_fn=<SliceBackward0>)


In [13]:
dataset=load_dataset("bloyal/deeploc")
print(dataset["train"].column_names)

README.md:   0%|          | 0.00/1.83k [00:00<?, ?B/s]

deeploc-train.parquet: reconstructing file:   0%|          |  0.00B / 12.9MB            

deeploc-train.parquet: downloading bytes:           |  0.00B            

deploc-val.parquet: reconstructing file:   0%|          |  0.00B / 1.70MB            

deploc-val.parquet: downloading bytes:           |  0.00B            

...

deeploc-test.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/22642 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2830 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2831 [00:00<?, ? examples/s]

['ACC', 'Kingdom', 'Membrane', 'Cytoplasm', 'Nucleus', 'Extracellular', 'Cell membrane', 'Mitochondrion', 'Plastid', 'Endoplasmic reticulum', 'Lysosome/Vacuole', 'Golgi apparatus', 'Peroxisome', 'Sequence']


In [14]:
location_columns = [
    "Cytoplasm",
    "Nucleus",
    "Extracellular",
    "Cell membrane",
    "Mitochondrion",
    "Plastid",
    "Endoplasmic reticulum",
    "Lysosome/Vacuole",
    "Golgi apparatus",
    "Peroxisome"
]

print(len(location_columns))

10


In [15]:
print(dataset["train"][0]["ACC"])
print(dataset["train"][0]["Kingdom"])
print(dataset["train"][0]["Sequence"][:50])

P33992
Metazoa
MSGFDDPGIFYSDSFGGDAQADEGQARKSQLQRRFKEFLRQYRVGTDRTG


In [17]:
print([dataset["train"][0][column] for column in location_columns])

[1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


In [18]:
train_data = dataset["train"].select(range(1000))
val_data = dataset["validation"].select(range(200))

print(len(train_data))
print(len(val_data))

1000
200


In [19]:
train_labels = torch.tensor([
    [row[column] for column in location_columns]
    for row in train_data
], dtype=torch.float32)

val_labels = torch.tensor([
    [row[column] for column in location_columns]
    for row in val_data
], dtype=torch.float32)

print(train_labels.shape)
print(val_labels.shape)

torch.Size([1000, 10])
torch.Size([200, 10])


In [22]:
train_sequences = list(train_data["Sequence"])
val_sequences = list(val_data["Sequence"])

train_tokens = tokenizer(
    train_sequences,
    padding=True,
    truncation=True,
    return_tensors="pt"
)

val_tokens = tokenizer(
    val_sequences,
    padding=True,
    truncation=True,
    return_tensors="pt"
)

print(train_tokens["input_ids"].shape)
print(val_tokens["input_ids"].shape)

torch.Size([1000, 5656])
torch.Size([200, 5497])


In [27]:
train_tokens={key:value.to(device) for key,value in train_tokens.items()}
val_tokens={key:value.to(device) for key,value in val_tokens.items()}

print(train_tokens["input_ids"].device)

cuda:0


In [25]:
for i,column in enumerate(location_columns):
  print(column,int(train_labels[:,i].sum()))

Cytoplasm 369
Nucleus 357
Extracellular 104
Cell membrane 141
Mitochondrion 90
Plastid 30
Endoplasmic reticulum 86
Lysosome/Vacuole 66
Golgi apparatus 40
Peroxisome 9


In [28]:
labels_per_protein=train_labels.sum(dim=1)
print(labels_per_protein.int().bincount())

tensor([  0, 747, 222,  25,   4,   2])


In [29]:
print(train_data[0]["Sequence"][:30])
print(tokenizer.decode(train_tokens["input_ids"][0][:35]))

MSGFDDPGIFYSDSFGGDAQADEGQARKSQ
<cls> M S G F D D P G I F Y S D S F G G D A Q A D E G Q A R K S Q L Q R R


In [30]:
print(train_data[0]["Sequence"][:10])
print(train_labels[0])

MSGFDDPGIF
tensor([1., 1., 0., 0., 0., 0., 0., 0., 0., 0.])


In [31]:
print(train_tokens["attention_mask"][0].sum())
print(train_tokens["input_ids"][0].shape)

tensor(736, device='cuda:0')
torch.Size([5656])


In [32]:
classifier=nn.Linear(320,10)
print(classifier)

Linear(in_features=320, out_features=10, bias=True)


In [33]:
for param in model.parameters():
  param.requires_grad= False
print(any(param.requires_grad for param in model.parameters()))

False


In [36]:
classifier= classifier.to(device)
print(next(classifier.parameters()).device)

cuda:0


In [37]:
batch_input_ids = train_tokens["input_ids"][:8]
batch_attention_mask = train_tokens["attention_mask"][:8]
batch_labels = train_labels[:8].to(device)

print(batch_input_ids.shape)
print(batch_labels.shape)

torch.Size([8, 5656])
torch.Size([8, 10])


In [38]:
with torch.no_grad():
    outputs = model(
        input_ids=batch_input_ids,
        attention_mask=batch_attention_mask
    )

batch_embeddings = outputs.last_hidden_state[:, 0, :]

print(batch_embeddings.shape)

torch.Size([8, 320])


In [39]:
batch_logits = classifier(batch_embeddings)

print(batch_logits.shape)
print(batch_logits[0])

torch.Size([8, 10])
tensor([ 0.1205, -0.2491, -0.2168, -0.0874, -0.1506,  0.3906,  0.0019,  0.0985,
         0.1383, -0.0340], device='cuda:0', grad_fn=<SelectBackward0>)


In [40]:
loss_fn = nn.BCEWithLogitsLoss()

batch_loss = loss_fn(batch_logits, batch_labels)

print(batch_loss.item())

0.7129122018814087


In [41]:
trainable_params = sum(
    p.numel() for p in classifier.parameters()
    if p.requires_grad
)

esm2_trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print("Classifier:", trainable_params)
print("ESM-2:", esm2_trainable_params)

Classifier: 3210
ESM-2: 0


In [43]:
optimizer= torch.optim.Adam(
    classifier.parameters(),
    lr=1e-3
)
print(optimizer)

Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


In [44]:
classifier.train()

for i in range(0, len(train_data), 8):
    batch_input_ids = train_tokens["input_ids"][i:i+8]
    batch_attention_mask = train_tokens["attention_mask"][i:i+8]
    batch_labels = train_labels[i:i+8].to(device)

    with torch.no_grad():
        outputs = model(
            input_ids=batch_input_ids,
            attention_mask=batch_attention_mask
        )

    batch_embeddings = outputs.last_hidden_state[:, 0, :]

    batch_logits = classifier(batch_embeddings)
    batch_loss = loss_fn(batch_logits, batch_labels)

    optimizer.zero_grad()
    batch_loss.backward()
    optimizer.step()

print("Training complete")

Training complete


In [50]:
torch.cuda.empty_cache()
print("CUDA cache cleared")

CUDA cache cleared


In [51]:
print("ESM-2 max position embeddings:", model.config.max_position_embeddings)
print("Tokenizer max length:", tokenizer.model_max_length)

ESM-2 max position embeddings: 1026
Tokenizer max length: 1000000000000000019884624838656


In [52]:
train_lengths = [len(tokenizer(seq)["input_ids"]) for seq in train_sequences]
val_lengths = [len(tokenizer(seq)["input_ids"]) for seq in val_sequences]

print("Train > 1026:", sum(length > 1026 for length in train_lengths))
print("Validation > 1026:", sum(length > 1026 for length in val_lengths))

Train > 1026: 112
Validation > 1026: 24


In [53]:
train_tokens = tokenizer(
    train_sequences,
    padding=True,
    truncation=True,
    max_length=1026,
    return_tensors="pt"
)

val_tokens = tokenizer(
    val_sequences,
    padding=True,
    truncation=True,
    max_length=1026,
    return_tensors="pt"
)

print(train_tokens["input_ids"].shape)
print(val_tokens["input_ids"].shape)

torch.Size([1000, 1026])
torch.Size([200, 1026])


In [54]:
train_tokens = {key: value.to(device) for key, value in train_tokens.items()}
val_tokens = {key: value.to(device) for key, value in val_tokens.items()}

print(train_tokens["input_ids"].device)
print(val_tokens["input_ids"].device)

cuda:0
cuda:0


In [55]:
classifier = nn.Linear(320, 10).to(device)

loss_fn = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    classifier.parameters(),
    lr=1e-3
)

print(classifier)

Linear(in_features=320, out_features=10, bias=True)


In [56]:
classifier.train()

total_loss = 0

for i in range(0, len(train_data), 8):
    batch_input_ids = train_tokens["input_ids"][i:i+8]
    batch_attention_mask = train_tokens["attention_mask"][i:i+8]
    batch_labels = train_labels[i:i+8].to(device)

    with torch.no_grad():
        outputs = model(
            input_ids=batch_input_ids,
            attention_mask=batch_attention_mask
        )

    batch_embeddings = outputs.last_hidden_state[:, 0, :]

    batch_logits = classifier(batch_embeddings)
    batch_loss = loss_fn(batch_logits, batch_labels)

    optimizer.zero_grad()
    batch_loss.backward()
    optimizer.step()

    total_loss += batch_loss.item()

print("Average training loss:", total_loss / (len(train_data) / 8))

Average training loss: 0.35173388278484347


In [57]:
val_embeddings_list = []

with torch.no_grad():
    for i in range(0, len(val_data), 8):
        batch_input_ids = val_tokens["input_ids"][i:i+8]
        batch_attention_mask = val_tokens["attention_mask"][i:i+8]

        outputs = model(
            input_ids=batch_input_ids,
            attention_mask=batch_attention_mask
        )

        batch_embeddings = outputs.last_hidden_state[:, 0, :]
        val_embeddings_list.append(batch_embeddings.cpu())

val_embeddings = torch.cat(val_embeddings_list)

print(val_embeddings.shape)

torch.Size([200, 320])


In [58]:
classifier.eval()

with torch.no_grad():
    val_logits = classifier(val_embeddings.to(device))

print(val_logits.shape)
print(val_logits[0])

torch.Size([200, 10])
tensor([-0.3681, -1.1231, -2.0068, -1.5822, -1.9437, -3.4016, -1.9264, -2.5811,
        -2.8807, -3.8663], device='cuda:0')


In [59]:
val_probs = torch.sigmoid(val_logits)

print(val_probs[0])
print("Maximum probability:", val_probs.max().item())
print("Minimum probability:", val_probs.min().item())

tensor([0.4090, 0.2454, 0.1185, 0.1705, 0.1252, 0.0322, 0.1271, 0.0704, 0.0531,
        0.0205], device='cuda:0')
Maximum probability: 0.53128981590271
Minimum probability: 0.00985619705170393


In [60]:
val_predictions = (val_probs >= 0.5).float()

print("Total positive predictions:", int(val_predictions.sum().item()))
print("Positive predictions per class:")
print(val_predictions.sum(dim=0))

Total positive predictions: 6
Positive predictions per class:
tensor([6., 0., 0., 0., 0., 0., 0., 0., 0., 0.], device='cuda:0')


In [62]:
true_positives=(val_predictions * val_labels.to(device)).sum(dim=0)
print("True positives per class:")
print(true_positives)

True positives per class:
tensor([3., 0., 0., 0., 0., 0., 0., 0., 0., 0.], device='cuda:0')


In [63]:
print("True positive labels per class:")
print(val_labels.sum(dim=0))

True positive labels per class:
tensor([66., 82., 18., 26., 18.,  5., 16., 11., 14.,  2.])


In [64]:
print("Average predicted probability per class:")
print(val_probs.mean(dim=0))

Average predicted probability per class:
tensor([0.4307, 0.2869, 0.1056, 0.1444, 0.1261, 0.0340, 0.0967, 0.0616, 0.0436,
        0.0187], device='cuda:0')


In [65]:
from sklearn.metrics import f1_score, accuracy_score

val_predictions = (val_probs >= 0.5).float()

f1 = f1_score(
    val_labels.numpy(),
    val_predictions.cpu().numpy(),
    average="micro"
)

accuracy = accuracy_score(
    val_labels.numpy(),
    val_predictions.cpu().numpy()
)

print("Validation F1:", f1)
print("Validation accuracy:", accuracy)

Validation F1: 0.022727272727272728
Validation accuracy: 0.01


In [66]:
baseline_results={
    "f1_micro":f1,
    "accuracy_exact_match":accuracy
}
print(baseline_results)

{'f1_micro': 0.022727272727272728, 'accuracy_exact_match': 0.01}


In [67]:
for param in model.parameters():
  param.requires_grad= True
print(any(param.requires_grad for param in model.parameters()))

True


In [68]:
optimizer = torch.optim.AdamW(
    [
        {"params": model.parameters(), "lr": 1e-5},
        {"params": classifier.parameters(), "lr": 1e-3}
    ]
)

print(optimizer)

AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 1e-05
    maximize: False
    weight_decay: 0.01

Parameter Group 1
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0.01
)


In [69]:
model.train()
classifier.train()

total_loss = 0

for i in range(0, len(train_data), 2):
    batch_input_ids = train_tokens["input_ids"][i:i+2]
    batch_attention_mask = train_tokens["attention_mask"][i:i+2]
    batch_labels = train_labels[i:i+2].to(device)

    outputs = model(
        input_ids=batch_input_ids,
        attention_mask=batch_attention_mask
    )

    batch_embeddings = outputs.last_hidden_state[:, 0, :]

    batch_logits = classifier(batch_embeddings)
    batch_loss = loss_fn(batch_logits, batch_labels)

    optimizer.zero_grad()
    batch_loss.backward()
    optimizer.step()

    total_loss += batch_loss.item()

print("Average fine-tuning loss:", total_loss / (len(train_data) / 2))

Average fine-tuning loss: 0.2616377043500543


In [70]:
model.eval()
classifier.eval()

val_embeddings_list = []

with torch.no_grad():
    for i in range(0, len(val_data), 8):
        batch_input_ids = val_tokens["input_ids"][i:i+8]
        batch_attention_mask = val_tokens["attention_mask"][i:i+8]

        outputs = model(
            input_ids=batch_input_ids,
            attention_mask=batch_attention_mask
        )

        batch_embeddings = outputs.last_hidden_state[:, 0, :]
        val_embeddings_list.append(batch_embeddings.cpu())

val_embeddings = torch.cat(val_embeddings_list)

print(val_embeddings.shape)

torch.Size([200, 320])


In [71]:
with torch.no_grad():
    val_logits_ft = classifier(val_embeddings.to(device))

val_probs_ft = torch.sigmoid(val_logits_ft)

print(val_logits_ft.shape)
print(val_probs_ft[0])

torch.Size([200, 10])
tensor([0.0875, 0.0382, 0.0255, 0.5957, 0.0479, 0.0093, 0.4168, 0.1330, 0.1498,
        0.0122], device='cuda:0')


In [72]:
val_predictions_ft = (val_probs_ft >= 0.5).float()

f1_ft = f1_score(
    val_labels.numpy(),
    val_predictions_ft.cpu().numpy(),
    average="micro"
)

accuracy_ft = accuracy_score(
    val_labels.numpy(),
    val_predictions_ft.cpu().numpy()
)

print("Fine-tuned Validation F1:", f1_ft)
print("Fine-tuned Validation accuracy:", accuracy_ft)

Fine-tuned Validation F1: 0.5557986870897156
Fine-tuned Validation accuracy: 0.26


In [73]:
from sklearn.metrics import f1_score

class_f1 = f1_score(
    val_labels.numpy(),
    val_predictions_ft.cpu().numpy(),
    average=None
)

for column, score in zip(location_columns, class_f1):
    print(column, round(score, 4))

Cytoplasm 0.4901
Nucleus 0.7105
Extracellular 0.8485
Cell membrane 0.5405
Mitochondrion 0.6667
Plastid 0.0
Endoplasmic reticulum 0.0
Lysosome/Vacuole 0.0
Golgi apparatus 0.0
Peroxisome 0.0


Biological Interpretation

A protein’s amino acid sequence contains information about its structure, interactions and ultimately where it functions inside a cell.

ESM-2 converts the amino acid sequence into numerical representations that capture patterns learned from large collections of protein sequences. The resulting 320-dimensional representation can therefore be viewed as a compact numerical description of sequence-level biological information.

For this task, the model uses the CLS representation as a summary of the complete protein sequence. The classification head then asks ten independent biological questions:

How likely is this protein to be associated with each cellular localization?

Importantly, these questions are independent. A protein can, for example, be associated with both the cytoplasm and nucleus, so the task cannot be treated as a single-choice classification problem.

The results also show an important biological/data-related limitation. Common localization classes such as extracellular, nucleus and mitochondrion were learned much better than rare classes such as peroxisome, Golgi apparatus and plastid.

This is partly a consequence of the limited training subset used during this controlled reconstruction. Rare localization classes contain very few examples, making it difficult for the model to learn reliable sequence patterns for them.

Therefore, the final results should be interpreted as evidence that ESM-2 can learn useful localization-related sequence information, while also demonstrating the importance of sufficient representation of rare biological classes.

In [74]:
print("Predicted positives per class:")
print(val_predictions_ft.sum(dim=0))

print("\nTrue positives per class:")
print(val_labels.sum(dim=0))

Predicted positives per class:
tensor([85., 70., 15., 11., 18.,  0.,  0.,  0.,  0.,  0.], device='cuda:0')

True positives per class:
tensor([66., 82., 18., 26., 18.,  5., 16., 11., 14.,  2.])


Technical Interpretation

The task was implemented as multi-label binary classification with 10 output neurons.

ESM-2 produces a 320-dimensional CLS representation for each protein:

Protein sequence → ESM-2 → 320-dimensional embedding

The classification head maps this representation to 10 logits:

320 → 10 logits

BCEWithLogitsLoss was used during training because each localization label represents an independent binary decision. During evaluation, sigmoid was applied to the logits and a threshold of 0.5 was used to convert probabilities into binary predictions.

Frozen ESM-2 baseline

In the first experiment, all ESM-2 parameters were frozen and only the classification head was trained.

Results:

* Micro-F1: 0.0227
* Exact-match accuracy: 0.01

The model produced very few positive predictions at the 0.5 threshold, indicating that the classifier alone was unable to effectively adapt the pretrained representation to the DeepLoc task.

Fine-tuned ESM-2

In the second experiment, ESM-2 was unfrozen and fine-tuned together with the classification head.

Results:

* Micro-F1: 0.5558
* Exact-match accuracy: 0.26

The large improvement demonstrates that adapting the pretrained ESM-2 representations to the specific protein localization task was critical.

The per-class results further show that the model learned the more represented classes substantially better, while several rare classes received zero F1 because the model produced no positive predictions for them at the 0.5 threshold.

This controlled comparison therefore provides evidence that the major limitation of the frozen baseline was not the basic classifier architecture or tensor pipeline, but the lack of task-specific adaptation of the ESM-2 representations.

What Went Wrong and What Was Corrected

The original project produced F1 = 0.00 and accuracy = 0.00, so the pipeline was reconstructed systematically instead of assuming a single cause.

1. Incorrect sequence-length handling

The ESM-2 model used in this project has:

max_position_embeddings = 1026

However, the tokenizer’s default model_max_length did not enforce this practical limit. Using:

truncation=True

without explicitly specifying max_length therefore allowed extremely long sequences to remain in the tokenized data.

Some proteins were tokenized to more than 5,000 tokens, far beyond the model’s supported sequence length.

Correction

The tokenizer was explicitly constrained:

max_length=1026

This ensured that sequences were truncated to the model’s supported maximum length and also resolved the severe GPU memory problems encountered during validation.

In the 1,000-protein training subset, 112 proteins (11.2%) exceeded 1026 tokens. In the 200-protein validation subset, 24 proteins (12%) exceeded this limit.

⸻

2. The frozen baseline was insufficient

The first controlled experiment kept ESM-2 completely frozen and trained only a 320 → 10 classification head.

This produced:

Micro-F1 = 0.0227

The result showed that the basic pipeline was functioning, but the fixed ESM-2 representation was not sufficiently adapted to the DeepLoc task.

Correction

ESM-2 was unfrozen and fine-tuned together with the classifier.

This increased the validation result to:

Micro-F1 = 0.5558

The large improvement demonstrates the importance of task-specific fine-tuning.

⸻

3. DeepLoc is highly imbalanced in the reconstruction subset

The 1,000-protein training subset contained very different numbers of examples for each localization class.

For example:

* Cytoplasm: 369
* Nucleus: 357
* Extracellular: 104
* Mitochondrion: 90
* Plastid: 30
* Golgi apparatus: 40
* Peroxisome: 9

The model consequently learned common classes much more effectively than rare ones.

After fine-tuning, the model predicted positive examples for the first five classes, but produced zero positive predictions for Plastid, Endoplasmic reticulum, Lysosome/Vacuole, Golgi apparatus and Peroxisome.

This explains their zero F1 scores on this small subset.

⸻

4. What was not found to be broken

Several possible sources of the original failure were explicitly checked:

* The classifier dimension was correct: 320 → 10
* The task was correctly treated as multi-label
* Label tensors had the correct shape: [N, 10]
* Predictions and labels had matching dimensions
* Sigmoid was used for independent label probabilities
* BCEWithLogitsLoss was appropriate for the task
* Sequence and label alignment was verified
* ESM-2 successfully produced [batch, sequence_length, 320] representations

Therefore, the investigation did not find evidence for a fundamental tensor-shape, label-alignment, classifier-dimension or sigmoid error.

Overall diagnosis

The reconstruction identified two important pipeline issues:

1. Sequence lengths were not explicitly constrained to ESM-2’s supported 1026-token limit.
2. The original approach did not sufficiently adapt ESM-2 representations to the DeepLoc task.

The corrected pipeline fixed the sequence-length problem and demonstrated the benefit of ESM-2 fine-tuning.

The remaining poor performance on rare classes is primarily associated with the limited and imbalanced reconstruction subset rather than a complete failure of the model pipeline.

Conclusion

This notebook successfully reconstructed and investigated the earlier ESM-2 + DeepLoc protein localization pipeline that had produced zero validation performance.

The controlled experiments showed a clear progression:

Experiment	Micro-F1	Exact-match Accuracy
Frozen ESM-2 baseline	0.0227	0.01
Fine-tuned ESM-2	0.5558	0.26

The investigation established that the core classification pipeline was functioning correctly. The major technical problems identified were improper sequence-length handling and the lack of effective task-specific adaptation when ESM-2 was kept frozen.

Explicitly limiting sequences to 1026 tokens corrected the sequence-length and memory problem. Fine-tuning ESM-2 then produced a substantial improvement in localization performance, increasing micro-F1 from 0.0227 to 0.5558 on the controlled validation subset.

The per-class results also revealed the effect of class imbalance. Common localization classes were learned considerably better than rare classes, several of which received zero F1 because the model made no positive predictions for them.

Final takeaway

The original zero-score result was therefore not evidence that ESM-2 or the DeepLoc task was fundamentally incompatible with the approach. Instead, the reconstruction showed that correct sequence handling, appropriate multi-label formulation, and task-specific fine-tuning are essential for obtaining meaningful performance.

This controlled reconstruction provides a validated foundation for the next stage: scaling the corrected pipeline to the full DeepLoc dataset and evaluating its generalization on unseen test proteins.